In [8]:
%cd ..
%reset -f

/home/remi


In [9]:
import pandas as pd
from src.features.blood_cell_metadata import BloodCellMetadata
from src.cfg import BARCELONA_DIR, MUNICH_DIR

df_barcelona = BloodCellMetadata(BARCELONA_DIR,"Barcelona").get_df()
df_Munich = BloodCellMetadata(MUNICH_DIR,"Munich").get_df()

min_samples = 70
counts_barcelona = df_barcelona["class_code"].value_counts()
counts_munich = df_Munich["class_code"].value_counts()
valid_classes = (counts_barcelona[counts_barcelona >= min_samples].index.intersection(counts_munich[counts_munich >= min_samples].index))
print("Classes conservées :", sorted(valid_classes))

df_barcelona = df_barcelona[df_barcelona["class_code"].isin(valid_classes)].copy()
df_Munich = df_Munich[df_Munich["class_code"].isin(valid_classes)].copy()
df_barcelona = (df_barcelona.groupby("class_code", group_keys=False).sample(n=64, random_state=132))
df_Munich = (df_Munich.groupby("class_code", group_keys=False).sample(n=64, random_state=132))
df_sample = pd.concat([df_barcelona,df_Munich],ignore_index=True)
print(df_sample.shape)

unique = df_sample['source'].unique()
source_mapping = {unique[0]: 0, unique[1]: 1}
df_sample['source'] = df_sample["source"].map(source_mapping).to_numpy()


Building metadata: 100%|██████████| 18365/18365 [00:00<00:00, 21536.18it/s]

Classes conservées : ['BAS', 'EBO', 'EOS', 'LYT', 'MON', 'NGB', 'NGS', 'PMO']
(1024, 6)


In [10]:
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df_sample, test_size=0.15, stratify=df_sample["source"], random_state=132)
train_df, val_df = train_test_split(train_df, test_size=0.15/0.85, stratify=train_df["source"], random_state=132)

In [11]:
import cv2
import numpy as np
from src.features.transform import *
from src.features.common import load_image

reference = load_image("data/PKG-AML-Cytomorphology_LMU/BAS/BAS_0008.tiff")

experiments = {
#     "baseline": [Compose([Resize((224, 224)),ToFloat32()], label=0),
#                  Compose([Resize((224, 224)),ToFloat32()], label=1)],
#     "crop": [Compose([Crop(),Resize((224, 224)),ToFloat32()], label=0),
#                  Compose([Crop(),Resize((224, 224)),ToFloat32()], label=1)],
#     "histogram equalizatio": [Compose([Crop(),Resize((224, 224)),HistogramEqualization(),ToFloat32()], label=0),
#                               Compose([Crop(),Resize((224, 224)),HistogramEqualization(),ToFloat32()], label=1)],
#     "clahe": [Compose([Crop(),Resize((224, 224)),Clahe(),ToFloat32()], label=0),
#               Compose([Crop(),Resize((224, 224)),Clahe(),ToFloat32()], label=1)],
#     "reinhard": [Compose([Crop(),Resize((224, 224)),Reinhard(reference),ToFloat32()], label=0),
#                  Compose([Crop(),Resize((224, 224)),ToFloat32()], label=1)],
    "macenko": [Compose([Crop(),Resize((224, 224)),Macenko(reference),ToFloat32()], label=0),
                 Compose([Crop(),Resize((224, 224)),ToFloat32()], label=1)],
#     "coral": [Compose([Crop(),Resize((224, 224)),Coral(reference),ToFloat32()], label=0),
#                  Compose([Crop(),Resize((224, 224)),ToFloat32()], label=1)],
    # "vahadane": [Compose([Crop(),Resize((224, 224)),Vahadane(reference),ToFloat32()], label=0),
    #              Compose([Crop(),Resize((224, 224)),ToFloat32()], label=1)],
   
    # "reinhard-coral": [Compose([Crop(),Resize((224, 224)),Reinhard(reference),Coral(reference),ToFloat32()], label=0),
    #                    Compose([Crop(),Resize((224, 224)),ToFloat32()], label=1)],reference = load_image("data/PKG-AML-Cytomorphology_LMU/BAS/BAS_0008.tiff")
    # "reinhard-vahadane": [Compose([Crop(),Resize((224, 224)),Reinhard(reference),Vahadane(reference),ToFloat32()], label=0),
    #                     Compose([Crop(),Resize((224, 224)),ToFloat32()], label=1)],
    # "CenterWeightedImage": [Compose([Crop(),Resize((224, 224)),CenterWeightedImage(),ToFloat32()], label=0),
    #              Compose([Crop(),Resize((224, 224)),CenterWeightedImage(),ToFloat32()], label=1)],
    # "Compress_Jpeg": [Compose([CompressJpeg(100),Crop(),Resize((224, 224)),ToFloat32()], label=0),
    #                     Compose([CompressJpeg(100),Crop(),Resize((224, 224)),ToFloat32()], label=1)],
}

In [ ]:
from src.features.blood_cell_dataset import BloodCellDataset
from src.features.CNN_binaire import CNNBinarie


results = []

for name, transforms in experiments.items():

    train_dataset = BloodCellDataset(metadata=train_df, label="source", transforms=transforms, batch_size=4, shuffle=False,label_transform="source")
    val_dataset = BloodCellDataset(metadata=val_df, label="source", transforms=transforms, batch_size=4, shuffle=False,label_transform="source")
    test_dataset = BloodCellDataset(metadata=test_df, label="source", transforms=transforms, batch_size=4, shuffle=False,label_transform="source")

    cnn = CNNBinarie(train_dataset,val_dataset, test_dataset, epochs=10)

    result = cnn.resultat()
    result["transform"] = name
    results.append(result)
    
results_df = pd.DataFrame(results)

Training:  20%|██        | 2/10 [00:20<01:20, 10.06s/epoch, acc=0.574, loss=0.652, val_acc=0.552]

In [6]:
display(results_df)
df_old = pd.read_csv("reports/domain_classifier.csv")
results_df = pd.concat([df_old,results_df],ignore_index=True)
results_df = results_df.drop_duplicates(subset=["transform"], keep="last")
results_df.to_csv("reports/domain_classifier.csv",index=False)

,accuracy,auc,precision_pbc,recall_pbc,f1_pbc,precision_munich,recall_munich,f1_munich,transform
0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,macenko


In [7]:
import plotly.graph_objects as go
import pandas as pd

results_df = pd.read_csv("reports/domain_classifier.csv")
# Tri par AUC croissante
plot_df = results_df.sort_values(
    by=["auc", "accuracy"],
    ascending=[False, False]
).copy()

# Couleur spéciale pour baseline
plot_df["color"] = plot_df["transform"].apply(
    lambda x: "baseline" if x == "baseline" else "transform"
)


fig = go.Figure()

# AUC
fig.add_trace(
    go.Bar(
        y=plot_df["transform"],
        x=plot_df["auc"],
        name="AUC",
        orientation="h",
        text=plot_df["auc"].round(3),
        textposition="auto",
        marker_color=[
            "red" if x == "baseline" else "steelblue"
            for x in plot_df["transform"]
        ],
        hovertemplate=(
            "<b>%{y}</b><br>"
            "AUC : %{x:.3f}<br>"
            "<extra></extra>"
        )
    )
)

# Accuracy
fig.add_trace(
    go.Scatter(
        y=plot_df["transform"],
        x=plot_df["accuracy"],
        mode="markers+text",
        name="Accuracy",
        text=plot_df["accuracy"].round(3),
        textposition="middle right",
        marker=dict(
            size=12,
            symbol="diamond"
        ),
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Accuracy : %{x:.3f}<br>"
            "<extra></extra>"
        )
    )
)


fig.update_layout(
    title="Impact des transforms sur le Domain Gap",
    xaxis_title="Score",
    yaxis_title="Transform",
    xaxis=dict(
        range=[0.5, 1.05]
    ),
    barmode="group",
    height=500,
    template="plotly_white"
)

fig.show()